# Kaarkhana on free Google Colab
Zero cost: no API key, no credit card. Everything runs inside this Colab VM (CPU is enough; a free T4 GPU only matters for the optional SAM 3 detector).

**Run top to bottom.** Runtime > Change runtime type > T4 GPU is optional.

## 1. Get the code
Upload `kaarkhana.zip` (the project zip) when prompted.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                       # choose kaarkhana.zip
zipfile.ZipFile(next(iter(up))).extractall('.')
os.chdir('kaarkhana')
!pip -q install "reportlab<5" "pillow<13" openpyxl opencv-python-headless

## 2. Pick the video
Use the bundled sample, or upload your own line video (.mp4).

In [ ]:
VIDEO = 'samples/sim_line_clip_150s.mp4'
# own video instead:
# up = files.upload(); VIDEO = next(iter(up))

## 3. Analyse (classical detector - free, CPU)

In [ ]:
from kaarkhana import pipeline as P
from kaarkhana.config import LineConfig

cfg = LineConfig.load('assets/sim10_line.json')   # copy + edit for your own line (stations, SMV, costing)
cfg.costing.wage_bdt_per_hr = 100                 # <- your numbers
cfg.costing.cm_usd_per_piece = 1.00

res = P.analyse(VIDEO, cfg)                       # anchors=((x1,y1),(x2,y2)) if plates are not auto-found
out = P.save_outputs(res, 'out')
print(res['story']['headline']); print(); print(res['story']['prescription'])

In [ ]:
import pandas as pd
display(res['table'].round(2)); display(res['recon'].round(2))
for lbl, ok, det in res['constraint'].evidence: print('YES' if ok else 'no ', lbl, '-', det)

In [ ]:
from google.colab import files
files.download(out['pdf'])                        # the signable report

## 4. (Optional) SAM 3 zero-shot bundle tracking - free
1. Open the SAM 3 model page on Hugging Face and accept the licence (free).
2. Create a read token at huggingface.co/settings/tokens.
3. Switch the runtime to **T4 GPU**, then run the cells below.

The checkpoint id is a parameter because it depends on what Meta has published; change `MODEL_ID` if you have access to a newer SAM 3.x checkpoint. This path is **not** exercised by the offline tests - verify it on your footage and compare with the classical result.

In [ ]:
!pip -q install -U transformers accelerate "pillow<13"
from huggingface_hub import login
import getpass; login(getpass.getpass('HF token: '))

In [ ]:
from kaarkhana.detect import SamDetector
MODEL_ID = 'facebook/sam3'
det = SamDetector(model_id=MODEL_ID)
res_sam = P.analyse(VIDEO, cfg, detector=det, sample_hz=2.0)   # fewer frames: SAM is heavier
print(res_sam['story']['headline'])

## 5. (Optional) Free local LLM to polish the wording
Runs a small open model inside Colab. Its text is accepted only if it contains no number that is not already in the measured facts.

In [ ]:
!pip -q install -U transformers accelerate
from kaarkhana.agent import LocalLLMAgent
res2 = P.analyse(VIDEO, cfg, agent=LocalLLMAgent())
print(res2['story'].get('narrative') or '(LLM text rejected by numeric guard - deterministic text used)')

## 6. Validate against known cycle times (simulated line only)

In [ ]:
from kaarkhana import validate
import json
print(json.dumps(validate.score(res, 'assets/sim10_ground_truth.json'), indent=1))